## Session 9, in class: perturbation, Blanchard and Kahn, and Klein

Computational Macro (HWS 2026), University of Mannheim. Thursday, 05.11.2026, 10:15 to 11:45.

This is the code behind the session 9 slides. Part 1 writes the stochastic growth model as one system of
equations and finds its steady state. Part 2 linearizes the system by finite differences. Part 3 solves the
linear system with the eigen-decomposition of Blanchard and Kahn, part 4 with the generalized Schur
decomposition of Klein, including a version of the model that only Klein can handle. Part 5 uses the solution:
impulse responses, a simulation against the exact path of the economy with the closed form, and the economy
with $\gamma = 2$ and $\delta = 0.1$, compared with a global solution.

**How to work with this notebook**

- Run the cells from top to bottom (`Shift + Enter`). After a restart of the kernel, start again at the top:
  later cells use what earlier cells define.
- There are **10 gaps**, marked `___`. The comment line `# Gap k: ...` directly above a gap says what goes in.
  Every gap is one line, and most are one line of the slides.
- Cells without a gap run as they are. Read them anyway: they are part of the method, and the comments
  explain each step.
- Each part ends with a **Check**: a number to compare with, from the slides. Do not move on before it fits.
- **Slides.** "Slide 19" and the like refer to the session 9 slides, by the number at the bottom of each slide.
  Keep them open next to the notebook.
- The solution notebook is posted after the session.

**Plan**

| Part | Content | Gaps | Slides | Time |
| --- | --- | --- | --- | --- |
| 1 | The model as one system $\mathbf{F}(\mathbf{x}_{t+1}, \mathbf{x}_t) = \mathbf{0}$, and its steady state | 1, 2 | 4 to 8 | 10 min |
| 2 | Linearization by finite differences: the matrices $\mathbf{A}$ and $\mathbf{B}$ | 3, 4 | 8 to 10 | 10 min |
| 3 | Blanchard and Kahn: eigenvalues, the count, $\mathbf{G}$ and $\mathbf{H}$ | 5, 6 | 14, 15 | 20 min |
| 4 | Klein: the generalized Schur decomposition, and a model with static equations | 7, 8 | 17 to 21 | 20 min |
| 5 | Impulse responses, simulation against the exact path, the economy with $\gamma = 2$, $\delta = 0.1$ | 9, 10 | 22 to 25 | 25 min |

**Notation, slides to code**

| Slides | Meaning | Code |
| --- | --- | --- |
| $\mathbf{x}_t = (k_t, z_t, c_t)$ | the variables of the model, states first | `x`, a vector of logs |
| $\hat{\mathbf{x}}_t = \ln \mathbf{x}_t - \ln \bar{\mathbf{x}}$ | log deviation from the steady state | `x - xss` |
| $\mathbf{s}_t = (k_t, z_t)$ | the states, $n_s = 2$ | `s`, `n_states` |
| $\mathbf{F}(\mathbf{x}_{t+1}, \mathbf{x}_t)$ | the equilibrium conditions, zero along an equilibrium path | `F(x_next, x, par)` |
| $\mathbf{A}$, $\mathbf{B}$ | $\mathbf{A} \, \mathbb{E}_t \hat{\mathbf{x}}_{t+1} = \mathbf{B} \, \hat{\mathbf{x}}_t$ | `A`, `B` |
| $\boldsymbol{\Gamma} = \mathbf{A}^{-1} \mathbf{B}$, $\lambda_i$ | the matrix of the difference equation and its eigenvalues | `Gamma`, `lam` |
| $\mathbf{S}$, $\mathbf{T}$, $\mathbf{Z}$ | the generalized Schur decomposition of $(\mathbf{A}, \mathbf{B})$ | `S`, `T`, `Z` |
| $\mathbf{G}$, $\mathbf{H}$ | $\hat{c}_t = \mathbf{G} \hat{\mathbf{s}}_t$ and $\hat{\mathbf{s}}_{t+1} = \mathbf{H} \hat{\mathbf{s}}_t$ | `G`, `H` |

**Toolbox: the few tools this notebook uses that are new**

| Tool | What it does | Example |
| --- | --- | --- |
| `exp.(x)`, `log.(x)` | elementwise on a vector: from logs to levels and back | the model in logs |
| `eigen(Gamma)` | eigenvalues (`.values`) and eigenvectors (`.vectors`, columns) of a square matrix | Blanchard and Kahn |
| `sortperm(v)` | the order that sorts `v` | stable eigenvalues first |
| `inv(V)`, `A \ B`, `B / A` | the inverse, $\mathbf{A}^{-1} \mathbf{B}$, and $\mathbf{B} \mathbf{A}^{-1}$ without forming the inverse | the formulas of slides 15 and 20 |
| `schur(A, B)`, `ordschur!(F, select)` | the generalized Schur decomposition, and the reordering that puts the selected eigenvalues first | Klein |
| `rank(A)` | the rank of a matrix | a singular $\mathbf{A}$ |
| `randn(MersenneTwister(seed), T)` | `T` draws from $N(0, 1)$, reproducible through the seed | the simulation |

Everything else is from the earlier notebooks: `A * B`, the dots `.+`, `.*`, `Base.@kwdef struct`.

### 0. Packages

Nothing new to install: `LinearAlgebra`, `Random`, and `Statistics` ship with Julia, and `Plots` is in the
course environment. The first cell activates the course environment in `Julia/` and installs every package
listed there that is missing on your laptop.

In [ ]:
import Pkg
Pkg.activate(joinpath(@__DIR__, ".."))   # the course environment in Julia/
Pkg.instantiate()                        # installs every package it lists that is missing on your laptop

In [ ]:
using Plots
using Printf
using LinearAlgebra
using Random
using Statistics

### 1. The model as one system of equations (gaps 1 and 2)

*Slides 4 to 8.*

The stochastic growth model: a household with CRRA utility $u(c) = c^{1 - \gamma} / (1 - \gamma)$ owns the
capital stock, output is $z_t k_t^\alpha$, capital depreciates at rate $\delta$, and log TFP follows
$\ln z_{t+1} = \rho \ln z_t + \varepsilon_{t+1}$ with $\varepsilon_{t+1} \sim N(0, \sigma^2)$. The three
equilibrium conditions, with $R_{t+1} = \alpha z_{t+1} k_{t+1}^{\alpha - 1} + 1 - \delta$:

$$
\begin{aligned}
k_{t+1} &= z_t k_t^\alpha + (1 - \delta) k_t - c_t && \text{resource constraint} \\
\ln z_{t+1} &= \rho \ln z_t + \varepsilon_{t+1} && \text{TFP} \\
u_c(c_t) &= \beta \, \mathbb{E}_t \bigl[ u_c(c_{t+1}) \, R_{t+1} \bigr] && \text{Euler equation}
\end{aligned}
$$

The parameters are $\alpha = 0.3$ and $\beta = 0.96$ as in sessions 1 and 2, $\rho = 0.9$, $\sigma = 0.02$.
Parts 1 to 4 and the first half of part 5 use $\gamma = 1$ and $\delta = 1$: log utility and full
depreciation. Then the policy is known in closed form, $k_{t+1} = \alpha \beta z_t k_t^\alpha$ and
$c_t = (1 - \alpha\beta) z_t k_t^\alpha$, and in logs that is exactly linear (slide 5). The second half of
part 5 sets $\gamma = 2$ and $\delta = 0.1$, where no closed form exists.

**The system.** The function `F(x_next, x, par)` takes the variables of tomorrow and today, each as the vector
$\mathbf{x} = (\ln k, \ln z, \ln c)$ in logs with the **states first**, and returns the three residuals. Each
residual is "log of the left side minus log of the right side", so that the residuals are dimensionless and
a residual of $0.01$ means one percent. Along an equilibrium path without shocks every residual is zero.
The steady state is the vector $\bar{\mathbf{x}}$ with $\mathbf{F}(\bar{\mathbf{x}}, \bar{\mathbf{x}}) = \mathbf{0}$:
$\bar z = 1$, and $k^*$ and $c^*$ from problem set 1 (slide 8).

In [ ]:
Base.@kwdef struct EconomicParameters
    α::Float64 = 0.3          # capital share
    β::Float64 = 0.96         # discount factor
    γ::Float64 = 1.0          # relative risk aversion, 1 is log utility
    δ::Float64 = 1.0          # depreciation rate, 1 is full depreciation
    ρ::Float64 = 0.9          # persistence of log TFP
    σ::Float64 = 0.02         # standard deviation of the TFP innovation
end

par = EconomicParameters()    # γ = 1, δ = 1

In [ ]:
function steady_state(par)
    # The steady state (k*, 1, c*) in levels (slide 8, the formula of problem set 1)
    # Gap 1: k* from the Euler equation at k = k': α k^(α - 1) + 1 - δ = 1 / β, solved for k
    k = ___
    c = k^par.α - par.δ * k                                           # the resource constraint at k = k'
    return [k, 1.0, c]
end

In [ ]:
function F(x_next, x, par)
    # The equilibrium conditions. x = (ln k, ln z, ln c) today, x_next the same tomorrow.
    # Returns the three residuals, each "log of the left side minus log of the right side" (slide 7)
    k, z, c = exp.(x)                                                           # today, in levels
    k1, z1, c1 = exp.(x_next)                                                   # tomorrow, in levels
    R1 = par.α * z1 * k1^(par.α - 1) + 1 - par.δ                                # gross return tomorrow
    return [
        log(k1) - log(z * k^par.α + (1 - par.δ) * k - c),                       # resource constraint
        log(z1) - par.ρ * log(z),                                               # TFP, without the shock
        # Gap 2: the Euler equation: log of u_c(c) = c^(-γ) minus log of β u_c(c') R'
        ___,
    ]
end

In [ ]:
xss = log.(steady_state(par))                                                   # the steady state, in logs
println("steady state k*, z, c*: ", round.(steady_state(par), digits = 4))
println("F at the steady state:   ", round.(F(xss, xss, par), digits = 10))

**Check (slide 8):** $k^* = 0.1689$ and $c^* = 0.4176$, and all three residuals are zero at the steady state
(up to $10^{-16}$). If the third residual is not zero, the Euler equation in gap 2 has a sign or a log wrong.

### 2. Linearization by finite differences (gaps 3 and 4)

*Slides 8 to 10.*

With $\hat{\mathbf{x}}_t = \ln \mathbf{x}_t - \ln \bar{\mathbf{x}}$ the first-order Taylor expansion of
$\mathbf{F}$ around the steady state is

$$
\mathbf{A} \, \mathbb{E}_t \hat{\mathbf{x}}_{t+1} = \mathbf{B} \, \hat{\mathbf{x}}_t,
\qquad
\mathbf{A} = \frac{\partial \mathbf{F}}{\partial \hat{\mathbf{x}}_{t+1}} \Big|_{\text{ss}}, \quad
\mathbf{B} = - \frac{\partial \mathbf{F}}{\partial \hat{\mathbf{x}}_{t}} \Big|_{\text{ss}} .
$$

Since `F` takes logs as arguments, the derivatives with respect to the log deviations are the derivatives
with respect to its arguments. We compute them by **central differences**: column $j$ of a Jacobian is
$[\mathbf{f}(\mathbf{x}_0 + h \mathbf{e}_j) - \mathbf{f}(\mathbf{x}_0 - h \mathbf{e}_j)] / 2h$ with
$h = 10^{-6}$, which is accurate to about $10^{-10}$. The function `jacobian(f, x0)` works for any vector
function `f` of one vector argument. `linearize` applies it twice: to $\mathbf{F}$ as a function of tomorrow's
variables with today's fixed at the steady state, and the other way round.

The expectation is gone: it passes through the linear expansion, and $\mathbb{E}_t \varepsilon_{t+1} = 0$.
Nothing in $\mathbf{A}$ and $\mathbf{B}$ depends on $\sigma$ (certainty equivalence, slide 8).

In [ ]:
function jacobian(f, x0; h = 1e-6)
    # The Jacobian of f at x0 by central differences: column j is df / dx_j (slide 8)
    n, m = length(x0), length(f(x0))
    J = zeros(m, n)
    for j in 1:n
        e = zeros(n)
        e[j] = h                                                              # a step in direction j only
        # Gap 3: the central difference: f one step up minus f one step down, over the length of the two steps
        J[:, j] = ___
    end
    return J
end

In [ ]:
function linearize(F, xss, par)
    # The matrices A and B of the linear system A E x' = B x, both evaluated at the steady state xss
    A = jacobian(x_next -> F(x_next, xss, par), xss)                          # derivative with respect to tomorrow
    # Gap 4: the derivative with respect to today, with the sign that moves it to the right-hand side (slide
    #        8)
    B = ___
    return A, B
end

In [ ]:
A, B = linearize(F, xss, par)
println("A ="); display(round.(A, digits = 4))
println("B ="); display(round.(B, digits = 4))
println("by hand: 1 / β = ", round(1 / par.β, digits = 4), "  1 / (α β) = ", round(1 / (par.α * par.β), digits = 4),
        "  (1 - α β) / (α β) = ", round((1 - par.α * par.β) / (par.α * par.β), digits = 4))

**Check (slide 10 and the backup slide with the numbers):** the third row of $\mathbf{A}$ is $(0.7, -1, 1)$, the first row of $\mathbf{B}$ is
$(1.042, 3.472, -2.472)$, the second row of $\mathbf{B}$ is $(0, 0.9, 0)$, and the third $(0, 0, 1)$. These are
the hand-derived entries, reproduced to $10^{-10}$. A first row of $\mathbf{B}$ with the opposite
sign means gap 4 lacks the minus.

### 3. Blanchard and Kahn: eigenvalues, the count, $\mathbf{G}$ and $\mathbf{H}$ (gaps 5 and 6)

*Slides 14 and 15.*

Here $\mathbf{A}$ is invertible, so the system can be written as
$\mathbb{E}_t \hat{\mathbf{x}}_{t+1} = \boldsymbol{\Gamma} \hat{\mathbf{x}}_t$ with
$\boldsymbol{\Gamma} = \mathbf{A}^{-1} \mathbf{B}$. The eigen-decomposition
$\boldsymbol{\Gamma} = \mathbf{V} \boldsymbol{\Lambda} \mathbf{V}^{-1}$ gives one eigenvalue per variable, each
a direction in which the system grows as $\lambda_i^t$. The solution is the one bounded path, and the
**counting rule** says when there is exactly one: as many stable eigenvalues ($|\lambda_i| < 1$) as states.

The recipe of slide 15, with the eigenvalues ordered so that the stable ones come first:
$\mathbf{W} = \mathbf{V}^{-1}$, its last $n_c$ rows $\mathbf{W}_u$ belong to the unstable eigenvalues, and
boundedness requires $\mathbf{W}_u \hat{\mathbf{x}}_t = \mathbf{0}$ in every period. Split $\mathbf{W}_u$ into
its state columns $\mathbf{W}_{us}$ and its control columns $\mathbf{W}_{uc}$, then

$$
\hat{c}_t = - \mathbf{W}_{uc}^{-1} \mathbf{W}_{us} \, \hat{\mathbf{s}}_t = \mathbf{G} \hat{\mathbf{s}}_t,
\qquad
\mathbf{H} = \boldsymbol{\Gamma}_{ss} + \boldsymbol{\Gamma}_{sc} \mathbf{G} .
$$

The function takes $\mathbf{A}$, $\mathbf{B}$, and the number of states, and returns $\mathbf{G}$,
$\mathbf{H}$, and the eigenvalues. Eigenvalues and eigenvectors can come out complex (they come in conjugate
pairs when they do), the result is real up to rounding, so the function returns the real parts.

In [ ]:
function solve_bk(A, B, n_states)
    # Blanchard and Kahn (slide 15): needs A invertible. Returns G (controls on states), H (law of motion of the
    # states), and the eigenvalues
    Γ = A \ B                                                                 # E x' = Γ x
    E = eigen(Γ)                                                              # eigenvalues, eigenvectors as columns
    order = sortperm(abs.(E.values))                                          # stable eigenvalues first
    λ, V = E.values[order], E.vectors[:, order]
    # Gap 5: the number of stable eigenvalues: how many have modulus below one
    n_stable = ___
    if n_stable != n_states
        println("Blanchard and Kahn: $n_stable stable eigenvalues for $n_states states")
    end
    W = inv(V)                                                                # rows are the left eigenvectors
    Wu = W[n_states+1:end, :]                                                 # the unstable rows must annihilate x
    # Gap 6: the controls on the states, from W_us s + W_uc c = 0 (slide 15, step 2); the state columns are
    #        the first n_states
    G = ___
    H = Γ[1:n_states, 1:n_states] + Γ[1:n_states, n_states+1:end] * G          # the first n_states rows of Γ
    return real(G), real(H), λ
end

In [ ]:
n_states = 2
G, H, λ = solve_bk(A, B, n_states)
println("eigenvalues: ", round.(λ, digits = 4))
println("G = ", round.(G, digits = 4))
println("H = "); display(round.(H, digits = 4))
println("largest difference to the closed form (α, 1): ", maximum(abs.(G .- [par.α 1.0])))

**Check (slides 14 and 15):** eigenvalues $0.3$, $0.9$, $3.472$, two stable for two states. $\mathbf{G} = (0.3, \, 1)$
and $\mathbf{H} = \bigl( \begin{smallmatrix} 0.3 & 1 \\ 0 & 0.9 \end{smallmatrix} \bigr)$: the closed form
$\hat{c}_t = \alpha \hat{k}_t + \hat{z}_t$ and $\hat{k}_{t+1} = \alpha \hat{k}_t + \hat{z}_t$ of slide 5, to
about $10^{-11}$, the precision of the finite differences. The three eigenvalues have names: $0.3 = \alpha$ is
how fast capital forgets its past, $0.9 = \rho$ is TFP, $3.472 = 1 / \alpha\beta$ is the unstable direction
that the choice of $\hat{c}_0$ switches off.

### 4. Klein: the generalized Schur decomposition (gaps 7 and 8)

*Slides 17 to 21.*

The generalized Schur (QZ) decomposition of the pair $(\mathbf{A}, \mathbf{B})$ writes
$\mathbf{Q}^{\mathsf{G}} \mathbf{A} \mathbf{Z} = \mathbf{S}$ and $\mathbf{Q}^{\mathsf{G}} \mathbf{B} \mathbf{Z} = \mathbf{T}$
with $\mathbf{S}$ and $\mathbf{T}$ upper triangular and $\mathbf{Q}$, $\mathbf{Z}$ unitary. The generalized
eigenvalues are $\lambda_i = t_{ii} / s_{ii}$, and a zero $s_{ii}$ is an infinite eigenvalue: a static equation,
which is eliminated like any other unstable direction. With the stable eigenvalues ordered first, slide 20 gives

$$
\mathbf{G} = \mathbf{Z}_{21} \mathbf{Z}_{11}^{-1}, \qquad
\mathbf{H} = \mathbf{Z}_{11} \mathbf{S}_{11}^{-1} \mathbf{T}_{11} \mathbf{Z}_{11}^{-1} ,
$$

where the index $1$ refers to the first $n_s$ rows or columns. Nothing in it inverts $\mathbf{A}$.

**One trap in the library call.** Both SciPy and Julia report the eigenvalues of the pair as
$\alpha_i / \beta_i$ with $\alpha_i = s_{ii}$ and $\beta_i = t_{ii}$, the reciprocal of our $\lambda_i$. Stable
in our sense, $|t_{ii}| < |s_{ii}|$, is therefore $|\beta_i| < |\alpha_i|$ in theirs, and that is the test to
hand to the sorting routine. The ready-made option "inside the unit circle" would select the wrong block.
The decomposition is done in complex arithmetic: complex eigenvalues come in pairs that the real version
keeps in $2 \times 2$ blocks, which would break the triangular reading. The result is real up to $10^{-16}$.

In [ ]:
function solve_klein(A, B, n_states)
    # Klein (2000), slide 20: the generalized Schur decomposition, A need not be invertible. Returns G, H, and the
    # eigenvalues (infinite ones appear as very large numbers)
    Fs = schur(complex(A), complex(B))                                        # A = Q S Z', B = Q T Z'
    # Gap 7: the test for a stable eigenvalue, |t_ii| < |s_ii|, in the library's names: β is t_ii, α is s_ii
    stable = ___
    n_stable = sum(stable)
    if n_stable != n_states
        println("Klein: $n_stable stable eigenvalues for $n_states states")
    end
    ordschur!(Fs, stable)                                                     # the stable eigenvalues first
    Z11, Z21 = Fs.Z[1:n_states, 1:n_states], Fs.Z[n_states+1:end, 1:n_states]   # the first n_states columns of Z, split
    S11, T11 = Fs.S[1:n_states, 1:n_states], Fs.T[1:n_states, 1:n_states]       # the stable block
    # Gap 8: the controls on the states (slide 20, step 2): Z21 times the inverse of Z11
    G = ___
    H = Z11 * (S11 \ T11) / Z11                                               # the law of motion (slide 20, step 3)
    eigs = Fs.β ./ Fs.α                                                       # our λ_i = t_ii / s_ii
    return real(G), real(H), eigs
end

In [ ]:
Gk, Hk, eigs = solve_klein(A, B, n_states)
println("eigenvalues: ", round.(eigs, digits = 4))
println("G = ", round.(Gk, digits = 4))
println("H = "); display(round.(Hk, digits = 4))
println("largest difference to Blanchard and Kahn: ", maximum(abs.(Gk .- G)), "  ", maximum(abs.(Hk .- H)))

**Check (slide 20):** the same eigenvalues (in the order the decomposition chose), the same $\mathbf{G}$ and
$\mathbf{H}$ as in part 3, to $10^{-15}$.

**A model that only Klein can handle.** Write the same economy the way it is written in practice (slide 17),
with output, investment, and the return as named variables:

$$
y_t = z_t k_t^\alpha, \qquad i_t = y_t - c_t, \qquad k_{t+1} = (1 - \delta) k_t + i_t, \qquad
R_{t+1} = \alpha \, y_{t+1} / k_{t+1} + 1 - \delta .
$$

Now $\mathbf{x}_t = (k_t, z_t, c_t, y_t, i_t)$, still two states, and two of the five equations contain no
variable dated $t + 1$. Those are two zero rows in $\mathbf{A}$. Nothing to fill in here: `F_ext` is `F` with
the two static equations added, `linearize` and `solve_klein` are the functions from above.

In [ ]:
function F_ext(x_next, x, par)
    # The same model with output y and investment i as named variables, x = (ln k, ln z, ln c, ln y, ln i)
    k, z, c, y, i = exp.(x)
    k1, z1, c1, y1, i1 = exp.(x_next)
    R1 = par.α * y1 / k1 + 1 - par.δ                                          # the return, from output and capital
    return [
        log(k1) - log((1 - par.δ) * k + i),                                   # capital accumulation
        log(z1) - par.ρ * log(z),                                             # TFP
        -par.γ * log(c) - log(par.β * c1^(-par.γ) * R1),                      # Euler equation
        log(y) - log(z) - par.α * log(k),                                     # production: static
        log(i) - log(y - c),                                                  # investment: static
    ]
end

function steady_state_ext(par)
    k, z, c = steady_state(par)
    y = k^par.α
    return [k, z, c, y, y - c]
end

In [ ]:
xss_ext = log.(steady_state_ext(par))
A_ext, B_ext = linearize(F_ext, xss_ext, par)
println("A_ext ="); display(round.(A_ext, digits = 4))
println("rank of A_ext: ", rank(A_ext), " of ", size(A_ext, 1))
G_ext, H_ext, eigs_ext = solve_klein(A_ext, B_ext, n_states)
println("eigenvalues: ", eigs_ext)
println("G_ext (rows: c, y, i on k, z) ="); display(round.(G_ext, digits = 4))
println("H_ext ="); display(round.(H_ext, digits = 4))

**Check (slide 18 and the backup slide on the extended system):** $\mathbf{A}$ has rank $3$ of $5$, so
$\mathbf{A}^{-1} \mathbf{B}$ does not exist and `solve_bk` would stop with a singular matrix. The eigenvalues are
$0.3$, $0.9$, $3.472$, and two of the order $10^{17}$, which is how the library writes $\infty$. Two stable
ones for two states, the count still fits. $\mathbf{G}$ is now $3 \times 2$ (consumption, output, and
investment on the two states), and all three rows read $(0.3, \, 1)$: with $\gamma = 1$, $\delta = 1$, $c$, $y$, and $i$ are
all proportional to $z k^\alpha$. $\mathbf{H}$ is the one from before.

### 5. Using the solution: impulse responses, simulation, and a second economy (gaps 9 and 10)

*Slides 22 to 25.*

With $\mathbf{G}$ and $\mathbf{H}$ in hand everything else is a recursion. An **impulse response** starts the
states at a shock, $\hat{\mathbf{s}}_0 = (0, \, 0.01)$ for a one percent TFP shock, and iterates
$\hat{\mathbf{s}}_{t+1} = \mathbf{H} \hat{\mathbf{s}}_t$ with no further shocks, the controls following as
$\hat{c}_t = \mathbf{G} \hat{\mathbf{s}}_t$. A **simulation** draws $\varepsilon_t \sim N(0, \sigma^2)$ from a
seeded generator and adds $\varepsilon_{t+1}$ to log TFP in every step. Both functions take $\mathbf{G}$,
$\mathbf{H}$, and the number of states, so that they work for the extended model and for the second economy alike.

In [ ]:
function impulse_response(G, H, n_states, T, shock)
    # The paths of states and controls for T periods after the shock vector hits the states in period 1 (slide 22)
    s = zeros(T, n_states)                                                    # row t is the state vector in period t
    s[1, :] = shock
    for t in 1:T-1
        # Gap 9: the law of motion: tomorrow's states from today's
        s[t+1, :] = ___
    end
    c = s * G'                                                                # the controls in every period, row t is G s_t
    return s, c
end

In [ ]:
T = 25
s_irf, c_irf = impulse_response(G, H, n_states, T, [0.0, 0.01])
println("k_hat in percent, periods 0 to 5: ", round.(100 .* s_irf[1:6, 1], digits = 4))
println("c_hat in percent, periods 0 to 5: ", round.(100 .* c_irf[1:6, 1], digits = 4))

In [ ]:
# The exact path with gamma = 1, delta = 1 in levels, k' = α β z k^α, after the same shock (slide 22)
kss, _, css = steady_state(par)
z_exact = exp.(par.ρ .^ (0:T-1) .* 0.01)                                        # ln z_t = ρ^t 0.01, no further shocks
k_exact = zeros(T)
k_exact[1] = kss
for t in 1:T-1
    k_exact[t+1] = par.α * par.β * z_exact[t] * k_exact[t]^par.α
end
c_exact = (1 - par.α * par.β) .* z_exact .* k_exact .^ par.α
println("largest difference between the exact path and the linear solution, k: ", maximum(abs.(log.(k_exact ./ kss) .- s_irf[:, 1])))

In [ ]:
tt = 0:T-1
p = plot(xlabel = "periods after the shock", ylabel = "percent deviation from steady state", size = (700, 320), legend = :topright, bottom_margin = 4Plots.mm, left_margin = 4Plots.mm)
plot!(p, tt, 100 .* s_irf[:, 2], color = :gray, lw = 1.5, label = "TFP")
plot!(p, tt, 100 .* s_irf[:, 1], lw = 1.8, color = 1, label = "capital, linear solution")
plot!(p, tt, 100 .* c_irf[:, 1], lw = 1.8, color = 2, label = "consumption, linear solution")
scatter!(p, tt, 100 .* log.(k_exact ./ kss), markershape = :circle, markercolor = :white, markerstrokecolor = 1, ms = 3, label = "exact path")
scatter!(p, tt, 100 .* log.(c_exact ./ css), markershape = :square, markercolor = :white, markerstrokecolor = 2, ms = 3, label = "")
hline!(p, [0], color = :black, lw = 0.5, label = "")

**Check (slide 22):** consumption jumps by one percent on impact, capital reaches $1.2$ percent after two
periods ($0$, $1.0$, $1.2$, $1.17$, $1.08$, $0.98$ percent), and both decay at $\rho$. The exact path and the
linear solution differ by about $10^{-11}$: with $\gamma = 1$ and $\delta = 1$ the linear solution is the closed form.

**Simulation.** Now with shocks in every period. The seed makes the draw reproducible, the same seed gives the
same path on every machine. The exact economy in levels is fed with the same innovations.

In [ ]:
function simulate(G, H, n_states, T, σ, seed)
    # A simulated path of T periods from the steady state, with innovations eps ~ N(0, σ^2) to log TFP (slide 23)
    rng = MersenneTwister(seed)                                               # seeded, so that the run is reproducible
    eps = σ .* randn(rng, T)
    s = zeros(T, n_states)
    for t in 1:T-1
        s[t+1, :] = H * s[t, :]
        # Gap 10: the innovation hits log TFP, the second state, in period t + 1
        ___
    end
    return s, s * G', eps
end

In [ ]:
function simulate_exact_bm(par, eps, kss)
    # The economy with γ = 1, δ = 1 in levels, k' = α β z k^α and c = (1 - α β) z k^α, with the innovations eps
    T = length(eps)
    k, z, c = zeros(T), ones(T), zeros(T)
    k[1] = kss
    for t in 1:T
        c[t] = (1 - par.α * par.β) * z[t] * k[t]^par.α
        if t < T
            k[t+1] = par.α * par.β * z[t] * k[t]^par.α
            z[t+1] = exp(par.ρ * log(z[t]) + eps[t+1])
        end
    end
    return k, z, c
end

In [ ]:
T_sim = 200
s_sim, c_sim, eps = simulate(G, H, n_states, T_sim, par.σ, 11)
k_ex, z_ex, c_ex = simulate_exact_bm(par, eps, kss)
println("standard deviations in percent, k_hat, c_hat, z_hat: ", round.(100 .* [std(s_sim[:, 1]), std(c_sim[:, 1]), std(s_sim[:, 2])], digits = 3))
println("largest difference between the exact path and the linear solution: ", maximum(abs.(log.(k_ex ./ kss) .- s_sim[:, 1])))

In [ ]:
p1 = plot(100 .* s_sim[:, 1], lw = 1.4, label = "capital, linear solution", ylabel = "percent", legend = :topright)
scatter!(p1, 100 .* log.(k_ex ./ kss), ms = 1.5, markerstrokewidth = 0, label = "exact path, same shocks")
plot!(p1, 100 .* s_sim[:, 2], color = :gray, lw = 0.8, label = "TFP")
hline!(p1, [0], color = :black, lw = 0.5, label = "")
p2 = plot(log.(k_ex ./ kss) .- s_sim[:, 1], color = :black, lw = 0.8, label = "", ylabel = "exact minus linear", xlabel = "period")
plot(p1, p2, layout = grid(2, 1, heights = [0.68, 0.32]), size = (700, 380), bottom_margin = 4Plots.mm, left_margin = 4Plots.mm)

**Check (slide 23):** with seed $11$ the standard deviations are about $5.9$ percent for capital, $6.0$ for
consumption, and $4.3$ for TFP (Julia's generator draws differently from Python's, which gives $5.1$, $5.1$, $3.7$ with the same seed, and another seed moves the first decimal). The exact path and the linear solution
differ by $10^{-11}$ over $200$ periods, with shocks of two percent: the replication is done.

**A second economy.** Two parameters change, $\gamma = 2$ and $\delta = 0.1$. The
code does not change at all. There is no closed form to compare with, so the yardstick is a **global
solution** of the deterministic version of the model ($\sigma = 0$), computed by the endogenous grid method of
week 5 on $400$ grid points. At first order the linear solution does not depend on $\sigma$ anyway, so the
deterministic policy is the right comparison for it. The solver is given. The Euler equation error $e(k)$ in
units of consumption is the one of problem set 1, now computed for the linear policy and for the global one.

In [ ]:
par2 = EconomicParameters(γ = 2.0, δ = 0.1)                                  # the second economy
xss2 = log.(steady_state(par2))
kss2, _, css2 = steady_state(par2)
A2, B2 = linearize(F, xss2, par2)
G2, H2, eigs2 = solve_klein(A2, B2, n_states)
println("steady state k*, z, c*: ", round.(steady_state(par2), digits = 4))
println("eigenvalues: ", round.(eigs2, digits = 4))
println("G = ", round.(G2, digits = 4))
println("H = "); display(round.(H2, digits = 4))

In [ ]:
T = 40
s1, c1 = impulse_response(G, H, n_states, T, [0.0, 0.01])                    # gamma = 1, delta = 1
s2, c2 = impulse_response(G2, H2, n_states, T, [0.0, 0.01])                  # gamma = 2, delta = 0.1
println("γ = 2, δ = 0.1: c_hat on impact ", round(100 * c2[1, 1], digits = 3), " percent, k_hat peaks at ", round(100 * maximum(s2[:, 1]), digits = 3),
        " percent in period ", argmax(s2[:, 1]) - 1)
tt = 0:T-1
pa = plot(title = "γ = 1, δ = 1", titlefontsize = 10, xlabel = "periods after the shock", ylabel = "percent deviation")
pb = plot(title = "γ = 2, δ = 0.1", titlefontsize = 10, xlabel = "periods after the shock")
for (p, s_, c_) in ((pa, s1, c1), (pb, s2, c2))
    plot!(p, tt, 100 .* s_[:, 2], color = :gray, lw = 1.2, label = "TFP")
    plot!(p, tt, 100 .* s_[:, 1], lw = 1.8, label = "capital")
    plot!(p, tt, 100 .* c_[:, 1], lw = 1.8, label = "consumption")
    hline!(p, [0], color = :black, lw = 0.5, label = "")
end
plot(pa, pb, layout = (1, 2), size = (800, 300), link = :y, bottom_margin = 4Plots.mm, left_margin = 4Plots.mm)

In [ ]:
function interp_extrap(x, xp, fp)
    # Linear interpolation of the points (xp, fp) at x, as interp1 of week 5, but with linear extrapolation beyond the ends
    out = similar(x)
    for (k, xq) in enumerate(x)
        i = clamp(searchsortedlast(xp, xq), 1, length(xp) - 1)               # the interval [xp[i], xp[i+1]] around xq
        w = (xq - xp[i]) / (xp[i+1] - xp[i])                                  # weight on the right end point, not clamped
        out[k] = (1 - w) * fp[i] + w * fp[i+1]
    end
    return out
end

function solve_global(par, kgrid; tol = 1e-10, maxit = 10_000)
    # The consumption policy c(k) of the deterministic growth model on kgrid, by the endogenous grid method of week 5:
    # kgrid plays tomorrow's capital, the Euler equation gives today's consumption, the resource constraint today's capital
    m_grid = kgrid .^ par.α .+ (1 - par.δ) .* kgrid                             # resources as a function of k, increasing
    c = m_grid .- kgrid                                                       # start: keep capital constant
    it = 0
    for outer it in 1:maxit
        R = par.α .* kgrid .^ (par.α - 1) .+ (1 - par.δ)                         # the return when k' is a grid point
        c_today = (par.β .* R .* c .^ (-par.γ)) .^ (-1 / par.γ)                # the Euler equation, c' = c(k') on the grid
        m_today = c_today .+ kgrid                                            # the resources that make this k' optimal
        k_today = interp_extrap(m_today, m_grid, kgrid)                       # the k that has those resources
        c_new = interp_extrap(kgrid, k_today, c_today)                        # back onto the fixed grid
        dist = maximum(abs.(c_new .- c))
        c = c_new
        dist < tol && break
    end
    return c, it
end

function euler_error(kgrid, c, par)
    # The Euler equation error of a consumption policy c on kgrid, in units of consumption (problem set 1), σ = 0
    k_next = kgrid .^ par.α .+ (1 - par.δ) .* kgrid .- c
    c_next = interp_extrap(k_next, kgrid, c)
    R = par.α .* k_next .^ (par.α - 1) .+ (1 - par.δ)
    return 1 .- (par.β .* R) .^ (-1 / par.γ) .* c_next ./ c
end

In [ ]:
kgrid = collect(range(0.1 * kss2, 2 * kss2, length = 400))                   # the grid of problem set 1
c_global, it = solve_global(par2, kgrid)
c_linear = css2 .* exp.(G2[1, 1] .* log.(kgrid ./ kss2))                      # the linear policy in levels, at z = 1
e_linear = euler_error(kgrid, c_linear, par2)
e_global = euler_error(kgrid, c_global, par2)
println("global solution: $it iterations")
println(" k / k*   c linear   c global   difference   |e| linear   |e| global")
for f in (0.1, 0.5, 0.9, 1.0, 1.1, 1.5, 2.0)
    i = argmin(abs.(kgrid .- f * kss2))
    @printf("%6.1f %10.4f %10.4f %+9.2f %%  %10.1e %11.1e\n", f, c_linear[i], c_global[i], 100 * (c_linear[i] / c_global[i] - 1), abs(e_linear[i]), abs(e_global[i]))
end

In [ ]:
pa = plot(kgrid ./ kss2, c_global, color = :black, lw = 1.6, label = "global solution", xlabel = "k / k*", ylabel = "consumption c(k)", legend = :bottomright)
plot!(pa, kgrid ./ kss2, c_linear, lw = 1.6, ls = :dash, label = "linear solution")
scatter!(pa, [1.0], [css2], color = 2, label = "steady state")
pb = plot(kgrid ./ kss2, abs.(e_linear) .+ 1e-16, lw = 1.6, yscale = :log10, ylims = (1e-8, 1), label = "linear solution", xlabel = "k / k*", ylabel = "Euler equation error |e(k)|", legend = :bottomright)
plot!(pb, kgrid ./ kss2, abs.(e_global) .+ 1e-16, color = :black, lw = 1.2, label = "global solution")
plot(pa, pb, layout = (1, 2), size = (800, 300), bottom_margin = 4Plots.mm, left_margin = 4Plots.mm)

**Check (slides 24 and 25):** $k^* = 2.921$, $c^* = 1.087$, eigenvalues $0.893$, $0.9$, $1.166$,
$\mathbf{G} = (0.399, \, 0.562)$. Consumption rises by $0.56$ percent on impact and capital peaks at $0.99$ percent
in period $9$. The global solution takes $82$ iterations. At $k = 0.9 \, k^*$ and $1.1 \, k^*$ the linear policy
is $0.02$ percent off the global one and its Euler equation error is about $10^{-4}$, at $0.5 \, k^*$ and
$2 \, k^*$ it is one percent off with an error of a few $10^{-3}$, at $0.1 \, k^*$ ten percent. The global
solution has errors of $10^{-6}$ everywhere (the interpolation on $400$ points). Exact at the steady state,
and the error grows with the square of the distance: that is what a first-order solution is.

**Experiment: the persistence of TFP.** Solve the economy with $\gamma = 2$ and $\delta = 0.1$ for $\rho \in \{0.5, 0.9, 0.99\}$ and plot
the impulse responses of capital and consumption next to each other. Look at what moves with $\rho$ and what
does not: the coefficient of $\hat{z}_t$ in $\mathbf{G}$, the period in which capital peaks, and the
eigenvalues. Then change $\rho$ with $\gamma = 1$ and $\delta = 1$ as well and check what happens to $\mathbf{G}$ there.

In [ ]:
pa = plot(title = "capital", titlefontsize = 10, xlabel = "periods after the shock", ylabel = "percent deviation")
pb = plot(title = "consumption", titlefontsize = 10, xlabel = "periods after the shock")
for ρ in (0.5, 0.9, 0.99)
    par_rho = EconomicParameters(γ = 2.0, δ = 0.1, ρ = ρ)
    Ap, Bp = linearize(F, log.(steady_state(par_rho)), par_rho)
    Gp, Hp, ep = solve_klein(Ap, Bp, n_states)
    sp, cp = impulse_response(Gp, Hp, n_states, 80, [0.0, 0.01])
    println("ρ = $ρ: G = ", round.(Gp, digits = 4), ", eigenvalues ", round.(sort(abs.(ep)), digits = 4),
            ", capital peaks at ", round(100 * maximum(sp[:, 1]), digits = 2), " percent in period ", argmax(sp[:, 1]) - 1)
    plot!(pa, 100 .* sp[:, 1], label = "ρ = $ρ")
    plot!(pb, 100 .* cp[:, 1], label = "ρ = $ρ")
end
hline!(pa, [0], color = :black, lw = 0.5, label = ""); hline!(pb, [0], color = :black, lw = 0.5, label = "")
plot(pa, pb, layout = (1, 2), size = (800, 300), link = :y, bottom_margin = 4Plots.mm, left_margin = 4Plots.mm)